# Бонусное задание. PySpark

Ноутбук запускается внутри контейнера `jupyter/pyspark-notebook`.
Датасеты доступны по пути `/home/jovyan/datasets/`.

In [1]:
import sys
# sys.path.insert(0, "/usr/local/spark/python")
# sys.path.insert(0, "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip")

## Задание 11. Загрузка данных и инспекция схемы

In [2]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("OlistAnalysis")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATASETS = "/home/jovyan/datasets"
# Ноутбук работает внутри docker-сети — хост warehouse, внутренний порт 5432.
JDBC_URL = "jdbc:postgresql://warehouse:5432/warehouse"
JDBC_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver",
}

In [3]:
# orders — из staging-слоя warehouse по JDBC (типы Postgres сохраняются).
orders_df = spark.read.jdbc(url=JDBC_URL, table="staging.orders", properties=JDBC_PROPS)
# customers и payments — из CSV, в DWH их нет.
customers_raw = spark.read.csv(f"{DATASETS}/olist_customers_dataset.csv", header=True)
payments_raw = spark.read.csv(f"{DATASETS}/olist_order_payments_dataset.csv", header=True)

for name, df in [("orders", orders_df), ("customers", customers_raw), ("payments", payments_raw)]:
    print(f"=== {name} ===")
    df.printSchema()
    df.show(3)

=== orders ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+--------

In [4]:
# # orders уже NOT NULL на уровне БД, чистим только CSV-источники.
# print(f"customers: {customers_raw.count()} → ", end="")
# customers_df = customers_raw.na.drop()
# print(customers_df.count())

# print(f"payments:  {payments_raw.count()} → ", end="")
# payments_df = payments_raw.na.drop()
# print(payments_df.count())

customers: 99441 → 99441
payments:  103886 → 103886


## Задание 12. Группировки и ранжирование

### Заказы по год-месяц и статусу

In [5]:
orders_with_ym = orders_df.withColumn(
    "order_year_month",
    F.date_format(F.col("order_purchase_timestamp"), "y-M"),
)

(
    orders_with_ym
    .groupBy("order_year_month", "order_status")
    .count()
    .withColumnRenamed("count", "No_of_orders_year_month")
    .orderBy("order_year_month")
).show(10)

+----------------+------------+-----------------------+
|order_year_month|order_status|No_of_orders_year_month|
+----------------+------------+-----------------------+
|         2016-10|    canceled|                     24|
|         2016-10|    invoiced|                     18|
|         2016-10|     shipped|                      8|
|         2016-10| unavailable|                      7|
|         2016-10|  processing|                      2|
|         2016-10|   delivered|                    265|
|         2016-12|   delivered|                      1|
|          2016-9|     shipped|                      1|
|          2016-9|    canceled|                      2|
|          2016-9|   delivered|                      1|
+----------------+------------+-----------------------+
only showing top 10 rows



### Рейтинг штатов по числу клиентов

In [6]:
rank_window = Window.orderBy(F.desc("No_of_customers_state"))

(
    customers_df
    .groupBy("customer_state")
    .count()
    .withColumnRenamed("count", "No_of_customers_state")
    .withColumn("rank", F.row_number().over(rank_window))
).show(27)

+--------------+---------------------+----+
|customer_state|No_of_customers_state|rank|
+--------------+---------------------+----+
|            SP|                41746|   1|
|            RJ|                12852|   2|
|            MG|                11635|   3|
|            RS|                 5466|   4|
|            PR|                 5045|   5|
|            SC|                 3637|   6|
|            BA|                 3380|   7|
|            DF|                 2140|   8|
|            ES|                 2033|   9|
|            GO|                 2020|  10|
|            PE|                 1652|  11|
|            CE|                 1336|  12|
|            PA|                  975|  13|
|            MT|                  907|  14|
|            MA|                  747|  15|
|            MS|                  715|  16|
|            PB|                  536|  17|
|            PI|                  495|  18|
|            RN|                  485|  19|
|            AL|                

## Задание 13. Оконные функции – повторные покупатели

### Клиенты с 3+ заказами

In [7]:
orders_customers = orders_df.join(customers_df, on="customer_id", how="left")

repeat_buyers = (
    orders_customers
    .groupBy("customer_unique_id")
    .count()
    .where(F.col("count") >= 3)
)

print(f"Клиентов с 3+ заказами: {repeat_buyers.count()}")

Клиентов с 3+ заказами: 252


### Дней между 1-м и 3-м заказом

In [8]:
w = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")
ranked = (
    orders_customers
    .join(repeat_buyers.select("customer_unique_id"), on="customer_unique_id", how="inner")
    .withColumn("order_rank", F.row_number().over(w))
)

first_orders = ranked.where(F.col("order_rank") == 1).select(
    "customer_unique_id",
    F.col("order_purchase_timestamp").alias("first_order_ts"),
)
third_orders = ranked.where(F.col("order_rank") == 3).select(
    "customer_unique_id",
    F.col("order_purchase_timestamp").alias("third_order_ts"),
)

(
    first_orders
    .join(third_orders, on="customer_unique_id")
    .withColumn("days_to_third", F.datediff("third_order_ts", "first_order_ts"))
    .agg(
        F.avg("days_to_third").alias("avg_days"),
        F.min("days_to_third").alias("min_days"),
        F.max("days_to_third").alias("max_days"),
    )
).show()

+------------------+--------+--------+
|          avg_days|min_days|max_days|
+------------------+--------+--------+
|131.54761904761904|       0|     633|
+------------------+--------+--------+

